In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.naive_bayes import GaussianNB

In [2]:
data = pd.read_csv(r"E:\Dataset\irrigation.csv")
print(data.head())

classes = data['crop'].unique()
print("\nClasses: ", classes)
print("Total Samples: ", len(data))

X = data[['moisture','temp']].values
y = data['pump'].values

     crop  moisture  temp  pump
0  cotton       638    16     1
1  cotton       522    18     1
2  cotton       741    22     1
3  cotton       798    32     1
4  cotton       690    28     1

Classes:  ['cotton']
Total Samples:  200


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [4]:
#model = GaussianNB()
#model.fit(X_train, y_train)

def compute_class_statistics(X, y):
    classes = np.unique(y)
    stats = {}
    for c in classes:
        X_c = X[y == c]
        stats[c] = {
            "mean": np.mean(X_c, axis=0),
            "var": np.var(X_c, axis=0),
            "prior": X_c.shape[0] / X.shape[0]
        }
        
    return stats

def gaussian_pdf(x, mean, var):
    exponent = np.exp(-(x - mean) ** 2 / (2 * var))
    return (1 / np.sqrt(2 * np.pi * var)) * exponent

def predict(X, stats):
    predictions = []
    for sample in X:
        posteriors = []
        
        for c, values in stats.items():
            prior = np.log(values["prior"])
            
            likelihood = np.sum(
                np.log(gaussian_pdf(sample, values["mean"], values["var"]))
            )
            
            posterior = prior + likelihood
            posteriors.append(posterior)
        
        predictions.append(np.argmax(posteriors))

    return np.array(predictions)

stats = compute_class_statistics(X_train, y_train)
print("Training Completed Successfully!")

Training Completed Successfully!


In [5]:
#y_pred = model.predict(X_test)

y_pred = predict(X_test, stats)

In [6]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.95


In [7]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:
[[ 7  2]
 [ 0 31]]


In [8]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.78      0.88         9
           1       0.94      1.00      0.97        31

    accuracy                           0.95        40
   macro avg       0.97      0.89      0.92        40
weighted avg       0.95      0.95      0.95        40

